In [ ]:
%load_ext autoreload
%autoreload 2

## Data Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [185]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

In [ ]:
df = pd.read_excel("../data/Training-data.xlsx")
print(df.head())

In [ ]:
test1 = pd.read_excel("../data/Test-data-1.xlsx")

In [ ]:
test2 = pd.read_excel("../data/Test-data-2.xlsx")

## Data Preparation (non-learned steps)

### Irrelevant Features

In [ ]:
df1 = df.drop(columns=["tcprtt"])

### Missing Values

In [ ]:
# No empty values for now

### Duplicated Values

In [ ]:
df2 = df1.drop(columns=["is_ftp_login"])

### Feature Attribute Selection

In [ ]:
df3 = df2.drop(columns=['dwin'])

### Feature Engineering

In [ ]:
# Empty for now

### Class Imbalance

In [ ]:
# Empty for now

## Cross Validation setup

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, ShuffleSplit

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=67
)

y = df3['label']
X = df3.drop(columns=['label'])


numeric_cols = X.select_dtypes(include=['number']).columns
categorical_cols = X.select_dtypes(include=['string']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), categorical_cols)
    ]
)

## k-NN Classifier

### k-NN: Build a pipeline 

In [ ]:
k_values = [1,3,7,11,17,21]

def make_knn_pipeline(k):
    return Pipeline([
        ("preprocessor", preprocessor),                 
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

### Step 3: Evaluate different k-values with CV

In [ ]:
cv_records = []
for k in k_values:
    print("Training on k = {}".format(k))
    pipe = make_knn_pipeline(k)
    scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    cv_records.append({"k": k, "mean_cv_acc": scores.mean(), "std_cv_acc": scores.std()})

### Step 4: Select the champion model

In [ ]:
cv_df = pd.DataFrame(cv_records).sort_values("mean_cv_acc", ascending=False).reset_index(drop=True)
print("\nCross-validation accuracy by k (training set):")
print(cv_df.to_string(index=False))

best_k = cv_df.loc[0, 'k']
print("Best k: {}".format(best_k))

scores = cross_validate(
    make_knn_pipeline(7),
    X,
    y,
    cv=skf,
    scoring=["accuracy", "precision", "recall", "f1"],
    n_jobs=-1
)

for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1"]:
    print(metric, scores[metric].mean(), scores[metric].std())

### Step 5: Retrain model with the full training dataset and evaluate on test datasets

In [ ]:
# pipe = make_knn_pipeline(best_k)
# pipe.fit(X, y)

# y_pred = pipe.predict(X_test)
# test_acc = accuracy_score(y_test, y_pred)

### Step 6: Display the classification scores and confusion matrix

In [ ]:
# print(f"Test accuracy (k={best_k}): {test_acc:.3f}")

# print("\nClassification report (test set):")
# print(classification_report(y_test, y_pred, digits=3))

### Step 7: Plot the confusion matrix for all k values for comparison

In [ ]:
# rows, cols = 2, 3
# fig, axes = plt.subplots(rows, cols, figsize=(15, 8))
# for idx, k in enumerate(k_values):
#     print("Training k = {}".format(k))
#     r, c = divmod(idx, cols)
#     pipe = make_knn_pipeline(k=k)
#     pipe.fit(X, y)
#     test1_pred = pipe.predict(test1)
#     test2_pred = pipe.predict(test2)
#     acc = accuracy_score(y_test, y_pred)

#     ConfusionMatrixDisplay.from_predictions(
#         y_test, y_pred,
#         cmap="Blues",
#         values_format="d",
#         ax=axes[r, c]
#     )
#     axes[r, c].set_title(f"k={k}  (Acc={acc:.2f})")

# plt.tight_layout()
# plt.show()